# OpenClaw — persistent Kaggle GPU deployment

This notebook restores encrypted OpenClaw state from GitHub, starts WhiteRabbitNeo on the Kaggle GPU, launches the OpenClaw Gateway, and prints the temporary Control UI URL and token. State is encrypted and pushed back every 15 minutes.

Add `GITHUB_TOKEN` and `OPENCLAW_BACKUP_KEY` under **Add-ons → Secrets** to enable restore and backup. Without them the session still runs, but state is ephemeral.


In [ ]:
import os
import pathlib
import subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
required = ("GITHUB_TOKEN", "OPENCLAW_BACKUP_KEY")
missing = []
for name in required + ("OPENCLAW_GATEWAY_TOKEN",):
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
    elif name in required:
        missing.append(name)
if missing:
    print("WARNING: Kaggle Secrets not configured: " + ", ".join(missing))
    print("WARNING: starting with fresh, ephemeral state; nothing will be restored or backed up.")
    print("WARNING: add them under Add-ons -> Secrets and rerun to enable persistence.")
else:
    print("Persistence secrets present: encrypted restore and backup enabled.")

repo = pathlib.Path("/kaggle/working/kaggle-sync")
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", "https://github.com/buurosaaqi-dev/kaggle-sync.git", str(repo)], check=True)

bundle = repo / "openclaw-kaggle"
subprocess.run(["bash", str(bundle / "scripts" / "run.sh")], cwd=bundle, env=os.environ.copy(), check=True)
